## Exploring the ODISSEI Knowledge Graph via SPARQL

Goal: extend the original query to also retrieve the **column name** and **description** of the tijdstip variable.
First, we explore what properties exist on variable and dataset nodes.

In [20]:
from SPARQLWrapper import SPARQLWrapper, JSON
import polars as pl

ENDPOINT = "https://api.kg.odissei.nl/datasets/odissei/odissei-kg/services/odissei-virtuoso/sparql"

def run_query(query: str) -> pl.DataFrame:
    """Run a SPARQL query and return results as a polars DataFrame."""
    sparql = SPARQLWrapper(ENDPOINT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results["results"]["bindings"]
    if not bindings:
        return pl.DataFrame()
    cols = results["head"]["vars"]
    data = {c: [row.get(c, {}).get("value", None) for row in bindings] for c in cols}
    return pl.DataFrame(data)

def wide_print(df, tbl_rows=3):
    with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000, tbl_rows=tbl_rows):
        print(df)

### Step 1: What properties does a variable node have?

Pick one dataset (ADOPTIEKINDEREN) and dump all predicates + values on its variables.

In [21]:
q_variable_props = """
prefix vi: <https://portal.odissei.nl/schema/variableInformation#>
prefix dct: <http://purl.org/dc/terms/>

SELECT ?var ?p ?o WHERE {
  ?dataset dct:alternative "ADOPTIEKINDEREN" .
  ?dataset vi:odisseiVariable ?var .
  ?var ?p ?o .
}
LIMIT 200
"""

var_props = run_query(q_variable_props)
wide_print(var_props)

shape: (200, 3)
┌──────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ var                                                                      ┆ p                                                                                     ┆ o                                                                                                           │
│ ---                                                                      ┆ ---                                                                                   ┆ ---                                                                                                         │
│ str                                                                      ┆ str                                                                               

### Step 2: What properties does a dataset node have?

Same idea: dump all predicates on the dataset node itself, to find the full dataset name.

In [22]:
q_dataset_props = """
prefix dct: <http://purl.org/dc/terms/>

SELECT ?p ?o WHERE {
  ?dataset dct:alternative "ADOPTIEKINDEREN" .
  ?dataset ?p ?o .
}
LIMIT 200
"""

ds_props = run_query(q_dataset_props)
wide_print(ds_props)

shape: (200, 2)
┌─────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────┐
│ p                                               ┆ o                                                                        │
│ ---                                             ┆ ---                                                                      │
│ str                                             ┆ str                                                                      │
╞═════════════════════════════════════════════════╪══════════════════════════════════════════════════════════════════════════╡
│ http://www.w3.org/1999/02/22-rdf-syntax-ns#type ┆ http://schema.org/Dataset                                                │
│ http://www.w3.org/1999/02/22-rdf-syntax-ns#type ┆ http://schema.org/Dataset                                                │
│ …                                               ┆ …                                          

### Step 3: Inspect the distinct predicates

Summarise the predicates we found to identify which ones hold the column name and description.

In [23]:
print("Variable predicates:")
wide_print(var_props["p"].unique().sort())
print()
print("Dataset predicates:")
wide_print(ds_props["p"].unique().sort())

Variable predicates:
shape: (6,)
Series: 'p' [str]
[
	"https://portal.odissei.nl/schema/variableInformation#odisseiConceptVariableDefinition"
	"https://portal.odissei.nl/schema/variableInformation#odisseiConceptVariableGroeppad"
	…
	"https://portal.odissei.nl/schema/variableInformation#odisseiConceptVariableValidFrom"
]

Dataset predicates:
shape: (32,)
Series: 'p' [str]
[
	"http://purl.org/dc/terms/alternative"
	"http://purl.org/dc/terms/creator"
	…
	"https://portal.odissei.nl/schema/variableInformation#odisseiVariable"
]


### Step 3b: What does each variable predicate look like for one concrete variable?

`odisseiConceptVariableName` gives a human-readable name, not a raw column ID. Let's check what `odisseiConceptVariableID` looks like — that might be the actual column name in the dataset.

### Step 4: Extend the original query

The variable node has two levels of properties:
- **Concept level** (`odisseiConceptVariable*`): abstract, shared across datasets
- **Variable level** (`odisseiVariable*`): concrete, specific to a dataset version

For the column name we want `vi:odisseiVariableName` (e.g., `ADODatumV`), not the concept name.
For the description we use `vi:odisseiVariableDefinition`.

In [24]:
q_extended = """
prefix schema: <http://schema.org/>
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>
prefix vi: <https://portal.odissei.nl/schema/variableInformation#>
prefix skos: <http://www.w3.org/2004/02/skos/core#>

SELECT DISTINCT ?datasetTitle ?altTitle ?tijdstipColName ?tijdstipVarLabel ?tijdstipDescription WHERE {
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset a schema:Dataset .
  ?dataset dct:alternative ?altTitle .
  ?dataset dct:title ?datasetTitle .

  # Person identifier variable
  ?dataset vi:odisseiVariable ?var1 .
  ?var1 vi:odisseiVariableVocabularyURI ?var1VocabURI .
  FILTER(?var1VocabURI = <https://w3id.org/odissei/cv/cbs/variableThesaurus/c650ad27af3f9e2b081c2b2f3698ae9eeb2a502f919b2f220d9f5b29afd6f337e>)

  # Tijdstip variable
  ?dataset vi:odisseiVariable ?tijdstipVar .
  ?tijdstipVar ?p ?varLabel .
  FILTER(isLiteral(?varLabel) && CONTAINS(LCASE(STR(?varLabel)), LCASE("Tijdstip")))
  ?tijdstipVar vi:odisseiVariableVocabularyURI ?tijdstipVocabURI .
  ?tijdstipVocabURI skos:prefLabel ?tijdstipVarLabel .

  # Column name (actual column in the dataset, e.g. ADODatumV)
  ?tijdstipVar vi:odisseiVariableName ?tijdstipColName .

  # Description of the tijdstip variable
  ?tijdstipVar vi:odisseiVariableDefinition ?tijdstipDescription .
}
ORDER BY ?altTitle
"""

extended = run_query(q_extended)
wide_print(extended.head(10))

shape: (10, 5)
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────┬────────────────────────────┬───────────────────────────────────────┬──────────────────────────────────────────────────────────────────────┐
│ datasetTitle                                                                                                                       ┆ altTitle                         ┆ tijdstipColName            ┆ tijdstipVarLabel                      ┆ tijdstipDescription                                                  │
│ ---                                                                                                                                ┆ ---                              ┆ ---                        ┆ ---                                   ┆ ---                                                                  │
│ str                                                  

In [25]:
extended.head()

datasetTitle,altTitle,tijdstipColName,tijdstipVarLabel,tijdstipDescription
str,str,str,str,str
"""Maandbedragen van personen die…","""AANVULVERGOEDPERSOONMNDBEDRAGB…","""EINDMNDBEDRAGAANVULVERGOED""","""Einde geldigheidsperiode attri…","""Datum (JJJJMMDD) tot welke de …"
"""Maandbedragen van personen die…","""AANVULVERGOEDPERSOONMNDBEDRAGB…","""AANVMNDBEDRAGAANVULVERGOED""","""Aanvang geldigheidsperiode att…","""Datum (JJJJMMDD) vanaf welke d…"
"""Kenmerken van adoptiekinderen""","""ADOPTIEKINDEREN""","""ADOJrV""","""Jaar van adoptie/vestiging""","""Als het jaar leeg is dan betre…"
"""Kenmerken van adoptiekinderen""","""ADOPTIEKINDEREN""","""ADODatumV""","""Datum van adoptie/vestiging""","""Als deze datum leeg is dan bet…"
"""Adult Education Survey- Onderz…","""AES""","""HATYEAR""","""Year when the respondent s hig…","""Dit betreft een speciale afgel…"


In [27]:
extended.shape

(579, 5)

Notes
- There's many more variables - which ones do we need? https://portal.odissei.nl/dataset.xhtml?persistentId=doi:10.57934/0B01E410802168E1
- why do we have now "only" 579? is this good or bad? b/c of missingness of the col names?